# A plain Python function is enough

The other notebooks in this directory wrap a framework. This one wraps
`def decide(payload): ...` -- and that is its point. It answers the two
questions a framework notebook cannot: what is the *minimum* to get an agent
into this market, and what does the adapter actually do for you.

It runs the same market twice through the IDENTICAL adapter: once with a
deterministic rule, once with a function that calls a language model. Nothing
about Tradefloor changes between the two runs -- not the validation, not the
execution, not the record -- which is the claim the whole integrations
subpackage rests on, shown rather than asserted.

## 1. Live calls, and the recording

Tradefloor's market is deterministic. A language model behind an API is not
-- the current SDK does not even take a temperature -- so two live runs give
two different agents and neither is reproducible.

This notebook **calls the model when it can and replays a recording when it
cannot**. The market re-executes for real either way; only the model's
answers come from the recording, keyed by a digest of the exact payload it
was sent.

The committed recording is a genuine claude-opus-5 run. Nothing in it was
written by hand.

In [1]:
import importlib.util
import json
import os
import sys
from pathlib import Path

import tradefloor as tf
from tradefloor.integrations.callable import callable_agent
from tradefloor.integrations.common import AdapterInfo, Transcript, digest

# The adapter is the production one, and the experiment's constants and both
# decision functions come from the module beside this notebook: one
# definition, so the two cannot drift into describing different experiments.
sys.path.insert(0, str(Path.cwd()))
import callable_agent as example

# Live requires an EXPLICIT opt-in on top of the key and the SDK. A
# credential in the environment is not consent to spend it, so replay is
# the default even when a live run is possible.
LIVE = example.can_run_live()
print(f"tradefloor {tf.__version__}")
print(f"mode        {'live, calling ' + example.LIVE_MODEL if LIVE else 'replay'}")
if not LIVE:
    missing = []
    if not os.environ.get(example.LIVE_OPT_IN_VAR):
        missing.append(f"{example.LIVE_OPT_IN_VAR}=1")
    if not example.have_live_key():
        missing.append(example.LIVE_KEY_VAR)
    if importlib.util.find_spec("anthropic") is None:
        missing.append("the 'anthropic' package")
    print(f"            live would need {', '.join(missing)}")

tradefloor 0.6.0
mode        replay
            live would need TRADEFLOOR_LIVE_EXAMPLES=1, ANTHROPIC_API_KEY


## 2. The market

Four synthetic names. Nothing about them is real -- the tickers, the sectors
and every fundamental are generated -- so neither agent can smuggle in
knowledge of a listed company, and the market has never appeared in any
training set.

In [2]:
roster = example.universe()
for ticker, sector, price, growth in example.ROSTER:
    print(f"{ticker:<10} {sector:<20} {price:>8.2f}   growth {growth:.2f}")
print(f"\nseed {example.SEED}, {example.DAYS} days, one decision per day")

TECH_A     technology             140.00   growth 0.30
TECH_B     technology              95.00   growth 0.22
BANK_A     financial_services      60.00   growth 0.05
STAPLE_A   consumer_staples        48.00   growth 0.02

seed 4242, 5 days, one decision per day


## 3. The rule is the whole integration

`mean_reversion` takes the serialized observation -- a plain dict -- and
returns a decision. That function is everything a user writes. The adapter
supplies the rest without being asked: the price memory behind the return
and volatility lines, the once-a-day cadence, the two-stage validation,
participation clipping, dust dropping, and a record joining every decision
to the exact input that produced it.

It sizes against both limits the payload states: `max_order_shares`, the
participation cap per order, and `buying_power`, the funding headroom under
the leverage cap. The first draft sized on participation alone -- at the
time, the payload named no other limit -- and every order it sent asked for
nine times the book's equity. The payload now states the funding side, and
the fix is one `min()`.

In [3]:
import inspect

source = inspect.getsource(example.mean_reversion)
print('def mean_reversion(payload):   # docstring elided; read the module')
print(source.split('"""')[-1])

def mean_reversion(payload):   # docstring elided; read the module

    book = payload["portfolio"]
    budget = 0.15 * book["net_worth"]
    if book["buying_power"] is not None:
        budget = min(budget, book["buying_power"])
    actions = []
    for asset in payload["assets"]:
        move = asset["return_5d"]
        if move is None:
            continue
        if move < -0.02:
            quantity = min(asset["max_order_shares"],
                           budget / asset["price"])
            actions.append({"symbol": asset["symbol"], "side": "BUY",
                            "quantity": quantity})
        elif move > 0.02 and asset["position"] > 0:
            actions.append({"symbol": asset["symbol"], "side": "SELL",
                            "quantity": asset["position"]})
    return {"actions": actions,
            "rationale": "five-day mean reversion, participation-sized"}



In [4]:
rule = callable_agent(example.mean_reversion)
rule_card = tf.evaluate({"rule": rule}, seed=example.SEED,
                        universe=roster, days=example.DAYS)["rule"]

print(f"trades {rule_card.trades}   pnl {rule_card.pnl:+,.0f}   "
      f"return {rule_card.return_pct:+.2f}%   rejected {rule_card.rejected}")
for entry in rule.record:
    acted = ", ".join(f"{a['side']} {a['quantity']:,.0f} {a['symbol']}"
                      for a in entry["decision"]["actions"]) or "hold"
    print(f"  day {entry['day']}  {acted}")

trades 3   pnl +5,067   return +0.51%   rejected 0
  day 0  hold
  day 1  BUY 1,612 TECH_B
  day 2  hold
  day 3  hold
  day 4  BUY 1,616 TECH_B, BUY 2,567 BANK_A


## 4. What the adapter refuses on your behalf

Two of the free guarantees, fired deliberately. A greedy function that sizes
to ten times the participation cap is **clipped**, and the clip is recorded
-- being unable to size a position says something about the agent, so the
trace carries it. A function that emits a `stop_loss` is **refused by
name**: this market executes market sweeps only, and a field dropped
silently would leave the agent believing it has protection that does not
exist.

In [5]:
from tradefloor.counterfactual import World
from tradefloor.integrations.common import DecisionError

def greedy(payload):
    asset = payload["assets"][0]
    return {"actions": [{"symbol": asset["symbol"], "side": "BUY",
                         "quantity": asset["max_order_shares"] * 10}]}

# max_leverage=None so the participation clip is what fires here, rather
# than the funding refusal arriving first.
clipped = callable_agent(greedy)
World(seed=example.SEED, universe=example.universe(), agent=clipped,
      max_leverage=None).run(days=1)
print("clipped:", clipped.record[0]["clipped"][0])
print()

def protected(payload):
    return {"actions": [{"symbol": "TECH_A", "side": "BUY",
                         "quantity": 100, "stop_loss": 120.0}]}

try:
    World(seed=example.SEED, universe=example.universe(),
          agent=callable_agent(protected)).run(days=1)
except DecisionError as refusal:
    print("refused:", str(refusal)[:220], "...")

clipped: TECH_A: asked for 1,000,000 shares, clipped to 100,000 (2.0% of average daily volume)

refused: action 0 carries unknown fields: stop_loss. Tradefloor executes market sweeps of signed share deltas -- there is no stop loss, no take profit and no time in force at the agent boundary -- so an unknown field cannot mean  ...


## 5. The same adapter, a model deciding

`llm_decide` is a function of the payload, exactly as `mean_reversion` is;
inside it happens to call claude-opus-5. The adapter cannot tell, and that
is the design. Live mode records every exchange; replay mode reads the
committed recording and never calls the function at all.

In [6]:
if LIVE:
    info = AdapterInfo(framework="callable", provider="anthropic",
                       model=example.LIVE_MODEL, agent_name="llm_decide",
                       instructions_digest=digest(example.MANDATE),
                       generation={"max_tokens": example.LIVE_MAX_TOKENS})
    llm = callable_agent(example.llm_decide, info=info, mode="live",
                         recorder=Transcript(), arm="live")
    llm.recorder.meta.update(llm.provenance())
    # Say what is about to be spent BEFORE spending it. The run cell below
    # is where the calls happen.
    print(f"live: llm_decide on {example.LIVE_MODEL}")
    print(f"the run below will make {example.DAYS} calls, one per simulated "
          f"day, up to {example.LIVE_MAX_TOKENS} output tokens each")
else:
    transcript = Transcript.load(example.FIXTURE)
    llm = callable_agent(example.llm_decide, mode="replay",
                         transcript=transcript, arm="replay")
    print(f"replaying {len(transcript)} recorded interactions from")
    print(f"  tests/fixtures/callable/{example.FIXTURE.name}\n")
    for field in ("framework", "provider", "model", "agent_name",
                  "generation", "instructions_digest",
                  "decision_every_steps", "decision_schema_version",
                  "recorded_utc"):
        print(f"  {field:<25} {transcript.meta.get(field)}")

replaying 5 recorded interactions from
  tests/fixtures/callable/five-days.json

  framework                 callable
  provider                  anthropic
  model                     claude-opus-5
  agent_name                llm_decide
  generation                {'max_tokens': 1500}
  instructions_digest       96a0f2163d6bbdd5
  decision_every_steps      6
  decision_schema_version   1
  recorded_utc              2026-08-31T02:14:43+00:00


## 6. What the model is sent

The payload, verbatim -- for the callable adapter there is no rendering
step between the serializer and the function, so the recorded input IS the
observation dict. The mandate rides as the system prompt. Note what the
observation carries and what it does not: prices, the book, both size
limits, the agent's own positions -- and no fair value, no attribution, no
future macro path. Those are the answer key, and they are used for scoring
on the other side of the wall.

In [7]:
first = llm.transcript.entries[0] if not LIVE else None

print("SYSTEM -- the mandate\n")
print(example.MANDATE)
if first is None:
    print("live run: the first payload is built at the first decision point")
else:
    print("USER -- the observation, verbatim\n")
    print(json.dumps(first["prompt"], indent=2)[:1800])
    print("  ...")

SYSTEM -- the mandate

You manage a portfolio inside a simulated financial market. The user message
is a JSON observation and is your only information: the tickers are
synthetic, so nothing you know about real securities applies.

Seek risk-adjusted returns while controlling downside. You are not required
to trade. Size every order within BOTH stated limits: max_order_shares per
order, and the portfolio's buying_power across the book.

Answer with a single JSON object and nothing else:
{"actions": [{"symbol": "<listed symbol>", "side": "BUY|SELL|HOLD",
"quantity": <shares>}], "rationale": "<one or two sentences>"}
An empty actions list means change nothing.

USER -- the observation, verbatim

{
  "step": 0,
  "day": 0,
  "steps_per_day": 6,
  "macro": {
    "federal_funds_rate": 0.025,
    "corporate_bond_yield": 0.0456,
    "vix": 15.0,
    "inflation_rate": 0.02,
    "cycle": "expansion"
  },
  "assets": [
    {
      "symbol": "TECH_A",
      "price": 140.0,
      "return_1d": null,

## 7. The run

In [8]:
llm_card = tf.evaluate({"llm": llm}, seed=example.SEED, universe=roster,
                       days=example.DAYS)["llm"]

print(f"trades {llm_card.trades}   pnl {llm_card.pnl:+,.0f}   "
      f"return {llm_card.return_pct:+.2f}%   rejected {llm_card.rejected}")
for entry in llm.record:
    acted = ", ".join(f"{a['side']} {a['quantity']:,.0f} {a['symbol']}"
                      for a in entry["decision"]["actions"]) or "hold"
    print(f"  day {entry['day']}  {acted}")
    print(f"          {entry['decision']['rationale'][:88]}")

trades 12   pnl +530   return +0.05%   rejected 0
  day 0  BUY 1,500 TECH_A, BUY 2,000 TECH_B, BUY 2,500 BANK_A, BUY 3,000 STAPLE_A
          Expansion regime with low VIX favors moderate long exposure, so I establish a diversifie
  day 1  BUY 1,000 TECH_B, BUY 1,000 BANK_A, HOLD 0 TECH_A, HOLD 0 STAPLE_A
          Expansion regime with low VIX supports staying invested, so I deploy part of the idle ca
  day 2  BUY 500 STAPLE_A, HOLD 0 TECH_A, HOLD 0 TECH_B, HOLD 0 BANK_A
          Low VIX and an expansion backdrop justify staying near fully invested, but with no stron
  day 3  BUY 1,000 STAPLE_A, BUY 500 BANK_A
          Expansion regime with low VIX and broad positive momentum supports staying invested; I d
  day 4  SELL 1,000 BANK_A, SELL 500 TECH_B, BUY 1,000 STAPLE_A
          Broad one-day drawdown led by BANK_A (-3.8%) and TECH_B, so I trim the weakest-momentum 


## 8. The boundary, made concrete

Two agents, one adapter class, one validation path, one market. The only
thing that changed between the columns below is who decided. An `if`
statement and a frontier model are interchangeable at this boundary, which
is what makes a comparison between them an experiment rather than two
different harnesses.

Do not read the P&L row as a ranking: a verdict from one seed measures the
seed at least as much as the agents. `tradefloor.rank` is the across-seed
answer.

In [9]:
assert type(rule) is type(llm), "one adapter class, or the claim is false"

print(f"{'':12} {'rule':>14} {'model':>14}")
print(f"{'adapter':<12} {type(rule).__name__:>14} {type(llm).__name__:>14}")
print(f"{'decisions':<12} {len(rule.record):>14} {len(llm.record):>14}")
for name in ("trades", "rejected"):
    print(f"{name:<12} {getattr(rule_card, name):>14} "
          f"{getattr(llm_card, name):>14}")
for name in ("turnover", "pnl"):
    print(f"{name:<12} {getattr(rule_card, name):>14,.0f} "
          f"{getattr(llm_card, name):>14,.0f}")
print(f"{'return':<12} {rule_card.return_pct:>13.2f}% "
      f"{llm_card.return_pct:>13.2f}%")

                       rule          model
adapter      CallableAgentAdapter CallableAgentAdapter
decisions                 5              5
trades                    3             12
rejected                  0              0
turnover            449,958      1,104,245
pnl                   5,067            530
return                0.51%          0.05%


## 9. Reproducing this

The recording is committed at `tests/fixtures/callable/five-days.json`.
Replaying needs no key, no network -- and no model function either: the
replayer below wraps a function that RAISES if called, and the run
completes without touching it, because replay never reaches the framework.

Change the observation mapping, the mandate or the market, and the digest
moves, the key goes missing, and the replay **refuses**, naming the step,
rather than answering the new question with an answer given to the old
one.

In [10]:
calls = []
def exploding(payload):
    calls.append(payload)
    raise AssertionError("replay called the function")

same = tf.evaluate(
    {"llm": callable_agent(exploding, mode="replay",
                           transcript=Transcript.load(example.FIXTURE))},
    seed=example.SEED, universe=example.universe(),
    days=example.DAYS)["llm"]

print(f"this notebook   {llm_card.pnl:+,.0f}")
print(f"fresh replay    {same.pnl:+,.0f}")
print(f"identical       {same.pnl == llm_card.pnl}")
print(f"model calls     {len(calls)}")
if LIVE:
    print("\nlive run: this compared a fresh live P&L against the committed")
    print("recording, so 'identical' above may honestly read False. To")
    print("refresh the fixture from this run:")
    print("    llm.recorder.save(example.FIXTURE)")

this notebook   +530
fresh replay    +530
identical       True
model calls     0


---

No third-party framework was involved, and that is the message: the adapter
contract *is* the integration. `callable_agent.py` beside this notebook is
the module of record; `tradefloor/integrations/common.py` documents
everything the adapter family shares; the other notebooks in this directory
show the same market traded through real frameworks, so the five can be
read side by side.